# 以 ollama 串接 langchain 執行 llama3-8b

In [1]:
# 設定必要的資源包和函式
# %pip install langchain chromadb pypdf fastembed
# %pip install aiofiles
# %pip install langchain_google_genai

In [2]:
import os
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy
import json

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain.schema.output_parser import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate
from langchain.vectorstores.utils import filter_complex_metadata
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory


if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

/Users/yanyifu/Documents/_Coding/LLM Predict Stock/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# 設定模型參數

In [3]:
model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0
chunk_size=1024
chunk_overlap=10

llm = ChatGoogleGenerativeAI(
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })


# f'''<|begin_of_text|><|start_header_id|>system<|end_header_id|> 

#     {system_prompt}<|eot_id|> 
#     <|start_header_id|>user<|end_header_id|>

#     {user_message}<|eot_id|>
#     <|start_header_id|>assistant<|end_header_id|>'''

# RAG

In [4]:
# 用來讀取文字檔，並轉成embedding
from typing import AsyncIterator, Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class CustomDocumentLoader(BaseLoader):
    """An example document loader that reads a file line by line."""

    def __init__(self, file_path: str) -> None:
        """Initialize the loader with a file path.

        Args:
            file_path: The path to the file to load.
        """
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:  # <-- Does not take any arguments
        """A lazy loader that reads a file line by line.

        When you're implementing lazy load methods, you should use a generator
        to yield documents one by one.
        """
        with open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1

    # alazy_load is OPTIONAL.
    # If you leave out the implementation, a default implementation which delegates to lazy_load will be used!
    async def alazy_load(
        self,
    ) -> AsyncIterator[Document]:  # <-- Does not take any arguments
        """An async lazy loader that reads a file line by line."""
        # Requires aiofiles
        # Install with `pip install aiofiles`
        # https://github.com/Tinche/aiofiles
        import aiofiles

        async with aiofiles.open(self.file_path, encoding="utf-8") as f:
            line_number = 0
            async for line in f:
                yield Document(
                    page_content=line,
                    metadata={"line_number": line_number, "source": self.file_path},
                )
                line_number += 1


In [5]:
# rag
class RAG:
    vector_store = None
    retriever = None
    chain = None

    def __init__(self):
        self.model = llm
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.prompt = PromptTemplate.from_template(
            """
            <s> [INST] 請判斷新聞對隔日開盤買進收盤賣出的獲利機率，如果可以獲利請回答#好 否則#不好 如果與股市無關則回答#無關 [/INST] </s> 
            [INST] 問題: {question} 
            文章: {context} 
            回答: [/INST]
            """
        )

    def ingest(self, pdf_file_path: str):
        docs = CustomDocumentLoader(file_path=pdf_file_path).load()
        chunks = self.text_splitter.split_documents(docs)
        chunks = filter_complex_metadata(chunks)
        # chunks.append(chunks)

        vector_store = Chroma.from_documents(documents=chunks, embedding=FastEmbedEmbeddings())
        self.retriever = vector_store.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={
                "k": 3,
                "score_threshold": 0.5,
            },
        )

        self.chain = ({"context": self.retriever, "question": RunnablePassthrough()}
                      | self.prompt
                      | self.model
                      | StrOutputParser())

    def ask(self, query: str):
        if not self.chain:
            return "Please, add a document first."

        return self.chain.invoke(query)

    def clear(self):
        self.vector_store = None
        self.retriever = None
        self.chain = None


In [6]:
# 測試rag是否能夠正常運行
import os

rag = RAG()
rag.clear()
rag.ingest(os.path.dirname(os.path.abspath(os.getcwd()))  + "/stock_news/3023/3023_news/20050129")

# response = rag.ask(f"請對{stock_id} {stock_names[idx]}的新聞對的隔日開盤買進收盤賣出進行判斷，如果可以獲利請回答#好 否則#不好 如果新聞內容與股市無關則回答#無關")
response = rag.ask(f"請找出對3023信邦的股價有影響的新聞資料")

print(response)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

回答: #好 



In [84]:
# rag 回應
def rag_response(ragfile, instruction):
    '''
        ragfile: string 檔案路徑
        instruction: string 問題指令
        
        return: string 回答
    '''
    text = ''
    try:
        rag = RAG()
        rag.clear()
        rag.ingest(ragfile)
        text = rag.ask(instruction)
        # print(text)
    except Exception as e:
        print(e)
        pass
    return text

In [96]:
def llm_response(filepath, question):
    ''' 
        filepath: string 檔案路徑
        question: string 問題指令
        
        return: string 回答
    '''
    with open(filepath, 'r') as f:
        news = f.read()

    prompt = f"""<s> [INST] 請判斷新聞對隔日開盤買進收盤賣出的獲利機率，如果可以獲利請回答#好 否則#不好 如果新聞內容與股市無關則回答#無關 [/INST] </s> 
            [INST] 問題: {question} 
            文章: {news} 
            回答: [/INST]
            """

    try:
        response = llm.invoke(prompt)
        # print(text)
        return response.content
    except Exception as e:
        print(e)
        pass
    
    return "無關"
    

In [86]:
# 轉換回答變成訊號
def response_to_signal(text):
    '''
        text: string 回答訊息
        
        return: int 回答訊號
    '''
    if text.find('非常不好') > 0:
        signal = -2
    elif text.find('非常好') > 0:
        signal = 2
    elif text.find('不好') > 0:
        signal = -1
    elif text.find('好') > 0:
        signal = 1
    else:
        signal = 0

    return signal

In [88]:
# 計算隔天當沖準確率（開盤買進收盤賣出or開盤賣出收盤回補）
def eval(signal_l, price):
    '''
        signal: list, [日期, 評分]
        price: dataframe, 股票價格

        return: float, float 準確率, 平均報酬率
    '''
    tp = 0
    tn = 0
    fp = 0
    fn = 0
    rtns = []   # 記錄當日沖銷的報酬率

    for idx, item in enumerate(signal_l):
        signal = float(item[1])
        index_location = price.index.get_loc(item[0])  # 獲取特定索引的位置
        if index_location == len(price.index) - 1: # 避免超出索引
            break
        next_index = price.index[index_location + 1]  # 獲取下一個索引的位置
        next_row = price.loc[next_index]  # 找到下一個索引的列
        open = float(next_row['Open'])
        close = float(next_row['Close'])
        
        if signal > 0:
            rtns.append(close/open)
            if close > open:
                tp += 1
            elif close < open:
                fp += 1
            else:   
                pass

        elif signal < 0:
            rtns.append(open/close)
            if close < open:
                tn += 1
            elif close > open:
                fn += 1
            else:   
                pass
    if (tp + tn + fp + fn)  == 0:
        accuracy = 0
    else:
        accuracy = (tp + tn) / (tp + tn + fp + fn) # 準確率
        print(tp, tn, fp, fn)

    if len(rtns) == 0:
        avg_rtn = 0
    else:
        avg_rtn = sum(rtns)/len(rtns)  # 這檔股票在這段時間的平均報酬率
        print(avg_rtn)
    return accuracy, avg_rtn

In [89]:
def write_output(content, folder_path):
    '''
        json_content: string, json格式的資料
        raw_data: list, 評分的資料
        folder_path: string, 資料夾路徑
    '''
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(content)
    print("寫入" + folder_path)

# RAG主程式

In [90]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ['2880', '2211', '2609']
stock_names = ['華南金', '長榮鋼', '陽明']

for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id} {stock_names[idx]}')
    qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                res = rag_response(folder_path + file_name, qurey)   # file_name, qurey
                sig = response_to_signal(res)    # file_name, qurey
                # print(f'{date}')
                break
        
        signal.append([date, sig]) # [日期, 訊號]
        # print(f'{date}: {sig}')

    acc, rtn = eval(signal, df_price) # 得到這檔股票的準確率和報酬率

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "system_prompt": system_prompt,
        "acc": acc,
        "rtn": rtn,
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    write_output(content=json_content, folder_path="eval_rag/output052701rag/" + stock_id + "/")

正在執行2880 華南金


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

20230103: 0
20230104: 0
20230105: 0


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

20230106: 1


...

20231227: 0


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

20231228: 0


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

20231229: 0
1 0 2 0
0.9973548852569026
寫入eval_rag/output052701rag/2609/


# 沒有RAG 當作對照組

In [101]:
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id} {stock_names[idx]}')
    prompt = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                res = llm_response(folder_path + file_name, prompt)
                print(res)
                sig = response_to_signal(res) 
                break
        
        signal.append([date, sig]) # [日期, 訊號]
        print(f'{date}: {sig}')

    acc, rtn = eval(signal, df_price) # 得到這檔股票的準確率和報酬率

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "acc": acc,
        "rtn": rtn,
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    write_output(content=json_content, folder_path="eval_rag/output norag/" + stock_id + "/")

正在執行2880 華南金
回答: #好 

新聞中提到華南金上漲逾 1%。 

20230103: 1
20230104: 0
20230105: 0
#無關 

這篇文章雖然提到了華南金，但沒有提供任何關於華南金的具體資訊，例如財務狀況、營運表現、政策利多或利空消息等，因此無法判斷它是否會對華南金的股價產生影響。 

20230106: 0
回答: #好 

華南金控 (2880) 在新聞中被提及，並且公布了 111 年 12 月份的營收資料，也提到了其在 2022 年的獲利情況，對華南金 (2880) 的股價有影響。 

20230109: 1
華南金 (2880-TW) 在這篇文章中被提及，并且有提到它在 2022 年逆勢成長，獲利較前年增加 0.58%。這表明華南金的財務表現相對良好，可能会对股价产生正面影响。

回答: #好 

20230110: 1
回答: #好 

**原因:** 

新聞提到了華南金在2022年全年稅後盈餘為173.04億元，較2021年微增1億元、0.5%，累計EPS為1.27元。而外資對華南金持保留態度，賣超華南金，這對華南金的股價不利。因此，隔日開盤買進收盤賣出的獲利機率較低。 

20230111: 1


KeyboardInterrupt: 

In [ ]:
# id = '2211'
# with open("eval_llama/output052601 01-23-39-234/" + id + "/detail.json") as f:
#     data = json.load(f)
#     print(data['raw_data'])
# df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + id + "twse.csv", dtype=str)
# df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
# for d in data['raw_data']:
#     print(d)
#     print(eval(d, df_price))
#     print()

FileNotFoundError: [Errno 2] No such file or directory: 'eval_llama/output052601 01-23-39-234/2211/detail.json'